12-rag-asnwers

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [39]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [14]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [ ]:
doc_idx

In [ ]:
q = ground_truth[10] 
q #Note, this is the "generated question" from the ground truth data, not the original question

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [ ]:
doc_idx[q['document']] #Note: this is the actual faq entry for docement '489dd1c9d9'

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [22]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [24]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp',
)

In [ ]:
q['question'] #Note, this is the "generated question" from the ground truth data, not the original question

'How do I join the Office Hours or live workshop if I don’t have the Zoom link?'

In [ ]:
answer = assistant.rag(q['question'])
#Ask the llm the "generated question" 

In [27]:
answer

'The Zoom link is only published to instructors, presenters, and TAs.\n\nIf you’re a student, join the session via:\n- **YouTube Live** on the DataTalksClub YouTube channel\n- **Slido** for questions, using the link pinned in the live chat\n- The **video URL** posted in the **announcements channel on Telegram and Slack** before the session starts\n\nDon’t post questions in chat, since they may be missed if the room is active.'

In [30]:
assistant.total_cost()

0.0012720000000000001

In [29]:
print(answer)

The Zoom link is only published to instructors, presenters, and TAs.

If you’re a student, join the session via:
- **YouTube Live** on the DataTalksClub YouTube channel
- **Slido** for questions, using the link pinned in the live chat
- The **video URL** posted in the **announcements channel on Telegram and Slack** before the session starts

Don’t post questions in chat, since they may be missed if the room is active.


In [32]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [ ]:
rag_result = {
    "question": q['question'], #Note, this is the "generated question" - generated from "answer_orig"
    "answer_llm": answer,
    "answer_orig": answer_orig, #Note, this is the original answer from the faq entry
    "document": doc_id,
}

rag_result

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': 'The Zoom link is only published to instructors, presenters, and TAs.\n\nIf you’re a student, join the session via:\n- **YouTube Live** on the DataTalksClub YouTube channel\n- **Slido** for questions, using the link pinned in the live chat\n- The **video URL** posted in the **announcements channel on Telegram and Slack** before the session starts\n\nDon’t post questions in chat, since they may be missed if the room is active.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as 

Function that compares the answer we get from llm (using generated question),
to the original answer

In [44]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [42]:
record = generate_rag_answer(q)
record

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': 'You can join without the Zoom link by watching the session on the DataTalksClub **YouTube Live** stream.\n\n- The Zoom link is only for instructors/presenters/TAs.\n- The **video URL** is posted in the **Telegram and Slack announcements channel** before the session starts.\n- If the session is live, you can also watch it on the **DataTalksClub YouTube Channel**.\n- Ask questions via **Slido**; the link is pinned in the chat when live.\n\nI don’t know if there’s any other way to access the Zoom link.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](h

In [45]:
assistant.total_cost()

0.0038610000000000003

In [46]:
assistant.reset_usage()

In [47]:
assistant.total_cost()

0.0

Now, we use "generate_rag_answer" to generate
"llm answer to generate-question" vs "original answer"
for ALL generated questions

then, we can evaluate all of the "llm answer to generated-questions"

In [49]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

In [ ]:
results[:10]

In [27]:
df_results = pd.DataFrame(results)

In [28]:
df_results.head()

,question,answer_llm,answer_orig,document
0,Is it okay to join the course late if I just f...,"Yes, you can still join the course late. If yo...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still take this course even if I missed ...,"Yes, you can still join if you missed the star...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,If I join after the course has already started...,"Yes, as long as you join while the course is s...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,Do I need to submit my project before submissi...,"Yes — to get the certificate, you need to subm...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,I’m a bit late to the course—what do I need to...,"To still earn the certificate, you need to:\n\...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [53]:
assistant.total_cost()


0.6332670000000004